In [1]:
%pip install mlflow scikit-learn pandas

  Using cached mlflow-3.9.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.9.0-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.9.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.3-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.4-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.85.0-py3-none-any.whl.metadata (40 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7 kB)
  

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2.- Crea un proyecto (Experimento)
experiment_name = "Proyecto_Diabetes_Notebook"
mlflow.set_experiment(experiment_name)

print(f"Experimento configurado: {experiment_name}")

c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/02/05 18:15:28 INFO mlflow.tracking.fluent: Experiment with name 'Proyecto_Diabetes_Notebook' does not exist. Creating a new experiment.


Experimento configurado: Proyecto_Diabetes_Notebook


In [3]:
# 4.- Usa el dataset (Diabetes para regresión)
data = load_diabetes()
X = data.data
y = data.target

# 6.- Prepara dataset de train y test (Splitting)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos preparados. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Datos preparados. Train shape: (353, 10), Test shape: (89, 10)


In [4]:
# Definimos hiperparámetros
params = {
    "fit_intercept": True,
    "copy_X": True
}

# 3.- Crea una ejecución (Run)
print("Iniciando entrenamiento y registro en MLflow...")

with mlflow.start_run() as run:
    # 5.- Utiliza un modelo (Regresión Lineal)
    model = LinearRegression(**params)
    
    # 6.- Entrena el modelo
    model.fit(X_train, y_train)
    
    # Predicciones para calcular métricas
    predictions = model.predict(X_test)
    
    # Cálculo de métricas
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    # 7.- Registra parámetros y métricas
    mlflow.log_params(params)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    
    # 8.- Guarda el modelo en mlflow
    mlflow.sklearn.log_model(model, "modelo_diabetes_regresion")
    
    # Guardamos el ID del run en una variable para usarla en la siguiente celda
    run_id_generado = run.info.run_id
    
    print(f"Modelo entrenado y registrado.")
    print(f"MSE: {mse:.2f}")
    print(f"Run ID: {run_id_generado}")

Iniciando entrenamiento y registro en MLflow...


2026/02/05 18:15:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Modelo entrenado y registrado.
MSE: 2900.19
Run ID: 6c8dabf1c19546e3842ea337ab6e7fa1


In [5]:
# 9.- Carga el modelo mlflow y realiza alguna predicción.

print(f"Cargando modelo desde: runs:/{run_id_generado}/modelo_diabetes_regresion")

# Cargar el modelo usando la URI dinámica
loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id_generado}/modelo_diabetes_regresion")

# Tomamos un dato de prueba (por ejemplo, el primero del set de test)
dato_nuevo = X_test[0].reshape(1, -1)

# Realizar predicción
prediccion = loaded_model.predict(dato_nuevo)

print("-" * 30)
print(f"Valor Real: {y_test[0]}")
print(f"Predicción: {prediccion[0]:.2f}")
print("-" * 30)

Cargando modelo desde: runs:/6c8dabf1c19546e3842ea337ab6e7fa1/modelo_diabetes_regresion


------------------------------
Valor Real: 219.0
Predicción: 139.55
------------------------------
